# Module 01 · Unit 03
# Boolean Algebra and Access Control

| | |
|---|---|
| **Estimated time** | 60–70 min |
| **Exercises** | Download PDF from the course repository |
| **Security thread** | Access Control & Policy Logic \[AC\] · Neural Network Architecture \[NN\] |
| **Refresher** | Module 01 · Unit 02 — Truth Tables |

---

## Learning Objectives

By the end of this notebook you will be able to:

1. State and apply the core Boolean algebra laws by name
2. Simplify a compound Boolean formula algebraically, citing each law used
3. Convert a formula to **Disjunctive Normal Form (DNF)** and **Conjunctive Normal Form (CNF)**
4. Use SymPy to verify equivalences and automate simplification
5. Identify logic gates as the hardware implementation of Boolean connectives
6. Analyse a four-variable ABAC policy, detect a policy bug algebraically,
   and confirm the blast radius with a truth table


## 🔣 Symbol Primer

No new logical connectives in this unit — all symbols carry over from Units 01–02.
Two new notation conventions appear for the first time:

| Notation | Meaning |
|---|---|
| $\mathtt{DNF}(\phi)$ | Disjunctive Normal Form of formula $\phi$ — OR of AND-terms |
| $\mathtt{CNF}(\phi)$ | Conjunctive Normal Form of formula $\phi$ — AND of OR-terms |
| $\oplus$ | Exclusive OR (XOR) — true when inputs **differ**; introduced in the gates section |

**The Boolean algebra laws** introduced in this unit use only $\neg$, $\wedge$, $\vee$,
$\equiv$, $\top$, $\bot$ — all previously defined.

---


## 1 · Boolean Algebra Laws

The laws below let you rewrite any Boolean expression into an equivalent one —
without constructing a truth table. Think of them as the algebra of logic:
the same way $2(x + 3) = 2x + 6$ uses the distributive law of arithmetic,
$(A \wedge T) \wedge (A \wedge \neg S)$ simplifies using idempotence and associativity.

| Law | $\wedge$ form | $\vee$ form |
|---|---|---|
| **Identity** | $P \wedge \top \equiv P$ | $P \vee \bot \equiv P$ |
| **Null (Domination)** | $P \wedge \bot \equiv \bot$ | $P \vee \top \equiv \top$ |
| **Idempotent** | $P \wedge P \equiv P$ | $P \vee P \equiv P$ |
| **Double Negation** | $\neg(\neg P) \equiv P$ | — |
| **Complement** | $P \wedge \neg P \equiv \bot$ | $P \vee \neg P \equiv \top$ |
| **Commutative** | $P \wedge Q \equiv Q \wedge P$ | $P \vee Q \equiv Q \vee P$ |
| **Associative** | $(P \wedge Q) \wedge R \equiv P \wedge (Q \wedge R)$ | $(P \vee Q) \vee R \equiv P \vee (Q \vee R)$ |
| **Distributive** | $P \wedge (Q \vee R) \equiv (P \wedge Q) \vee (P \wedge R)$ | $P \vee (Q \wedge R) \equiv (P \vee Q) \wedge (P \vee R)$ |
| **De Morgan 1** | $\neg(P \wedge Q) \equiv \neg P \vee \neg Q$ | — |
| **De Morgan 2** | $\neg(P \vee Q) \equiv \neg P \wedge \neg Q$ | — |
| **Absorption** | $P \wedge (P \vee Q) \equiv P$ | $P \vee (P \wedge Q) \equiv P$ |

> **How to use this table.** When simplifying, scan for patterns that match a
> law's left side and replace with the right side. Chain multiple steps.
> Every step preserves equivalence — the truth table never changes.


## 2 · Algebraic Simplification — Worked Examples

### Example 1 — Eliminating a Redundant Clause

**Expression:** $(A \wedge T) \wedge (A \wedge \neg S)$

$$\begin{align}
(A \wedge T) \wedge (A \wedge \neg S)
  &\equiv A \wedge T \wedge A \wedge \neg S
    &&\text{(Associative)} \\
  &\equiv A \wedge A \wedge T \wedge \neg S
    &&\text{(Commutative)} \\
  &\equiv A \wedge T \wedge \neg S
    &&\text{(Idempotent: } A \wedge A \equiv A\text{)}
\end{align}$$

The doubled $A$ clause collapses to a single one — no change in who is admitted.

---

### Example 2 — The Vacuous MFA Exemption

This is the key example for this unit. Suppose a developer writes a policy
intended to mean *"admin with MFA, or admin with a special MFA exemption"*:

$$\phi \;=\; (A \wedge T \wedge \neg S \wedge M) \;\vee\; (A \wedge T \wedge \neg S \wedge \neg M)$$

Step-by-step simplification:

$$\begin{align}
\phi
  &\equiv A \wedge T \wedge \neg S \wedge M
    \;\vee\; A \wedge T \wedge \neg S \wedge \neg M
    &&\text{(written out)} \\[4pt]
  &\equiv A \wedge T \wedge \neg S \wedge (M \vee \neg M)
    &&\text{(Distributive — factor out } A \wedge T \wedge \neg S\text{)} \\[4pt]
  &\equiv A \wedge T \wedge \neg S \wedge \top
    &&\text{(Complement: } M \vee \neg M \equiv \top\text{)} \\[4pt]
  &\equiv A \wedge T \wedge \neg S
    &&\text{(Identity: } P \wedge \top \equiv P\text{)}
\end{align}$$

**The MFA check has been completely removed.**

What the developer intended as an exemption for *some* admins became an exemption
for *all* admins. The complement law is ruthless: $M \vee \neg M$ covers every
possible MFA state, so the conjunction with $A \wedge T \wedge \neg S$ adds nothing.

This is a real class of security policy bug. Boolean algebra surfaces it in four
algebraic steps; a code review might not catch it at all.


## 3 · Normal Forms: DNF and CNF

### Why Normal Forms?

Any Boolean formula can be rewritten into a **standard structure** that is
mechanically recognisable and comparable. Two normal-form expressions with the
same variables are equivalent if and only if they are syntactically identical
(after sorting). This makes equivalence-checking algorithmic.

### Disjunctive Normal Form (DNF) — "OR of ANDs"

A formula is in **DNF** when it is a disjunction ($\vee$) of one or more
**conjunctive terms**, each of which is a conjunction ($\wedge$) of literals.
A **literal** is either an atomic proposition or its negation.

$$\mathtt{DNF}: \quad (L_1 \wedge L_2 \wedge \cdots) \;\vee\; (L_3 \wedge L_4 \wedge \cdots) \;\vee\; \cdots$$

Each conjunctive term corresponds to one **satisfying row** in the truth table.

**Security reading.** DNF answers: *"Under exactly which conditions is access
granted?"* Each AND-term is one specific combination of attributes that unlocks
the gate. Reading off the DNF of a policy tells you every path to access.

### Conjunctive Normal Form (CNF) — "AND of ORs"

A formula is in **CNF** when it is a conjunction ($\wedge$) of one or more
**clauses**, each of which is a disjunction ($\vee$) of literals.

$$\mathtt{CNF}: \quad (L_1 \vee L_2 \vee \cdots) \;\wedge\; (L_3 \vee L_4 \vee \cdots) \;\wedge\; \cdots$$

Each clause corresponds to one **falsifying row** in the truth table —
it rules out one specific input combination.

**Security reading.** CNF answers: *"Under which conditions must access be
denied?"* Each OR-clause is a constraint that at least one denial condition
must hold. CNF is the form used by SAT solvers, which underpin many modern
formal verification tools.

### Converting to DNF

The mechanical procedure: for each row in the truth table where the formula is
True, write the **minterm** — the conjunction of every variable in positive form
if the variable is T in that row, negated if F. The DNF is the disjunction of
all minterms.


## 4 · Logic Gates — Boolean Algebra in Hardware

The five logical connectives are not only abstract symbols — they are physically
implemented as **logic gates** in every CPU, GPU, and TPU. A gate takes electrical
signals (high voltage = T, low voltage = F) and produces an output signal according
to exactly the truth tables you have built.

| Gate | Symbol | Boolean operation | Output T when... |
|---|---|---|---|
| NOT | $\neg$ | $\neg A$ | Input is F |
| AND | $\wedge$ | $A \wedge B$ | Both inputs are T |
| OR | $\vee$ | $A \vee B$ | At least one input is T |
| NAND | $\neg(\cdot \wedge \cdot)$ | $\neg(A \wedge B)$ | NOT both T — the universal gate |
| NOR | $\neg(\cdot \vee \cdot)$ | $\neg(A \vee B)$ | Both inputs are F |
| XOR | $\oplus$ *(exclusive or)* | $A \oplus B$ | Inputs **differ** |

### The NAND Gate — A Universal Building Block

The NAND gate is **functionally complete**: any Boolean function can be built
from NAND gates alone. This is why early CPUs were implemented entirely in NAND
logic — fewer distinct chip designs to manufacture.

$$\neg A \;\equiv\; A \mathbin{\text{NAND}} A \qquad\qquad
  A \wedge B \;\equiv\; \neg(A \mathbin{\text{NAND}} B)$$

### Connection to Neural Networks

A **TPU Multiply-Accumulate (MAC) unit** performs the core operation of a neural
network forward pass:

$$\text{output} = \sum_i w_i \cdot x_i$$

Each $w_i \cdot x_i$ multiplication is implemented at the gate level as AND
operations over the binary representations of $w_i$ and $x_i$. The accumulator
uses XOR and carry chains (half-adders: XOR for sum, AND for carry).

The Boolean algebra you are learning in this module is the formal language of the
hardware that runs the AI models you will study in Modules 04–06. The same laws
that simplify an access control policy also let hardware engineers minimise the
gate count of a MAC unit.


---

## 🔐 Security Bridge &nbsp;·&nbsp; \[AC\] \[NN\]

| Concept | \[AC\] — Access Control | \[NN\] — Neural Network Hardware |
|---|---|---|
| **Boolean identity** | Drop a $\wedge\,\top$ clause — no change to access | Eliminate redundant gate wiring |
| **Idempotent** | Duplicate role check collapses to one | Duplicate signal in a circuit simplified away |
| **Complement** | $M \vee \neg M \equiv \top$ — MFA check erased | Complementary signals cancel in combinational logic |
| **Distributive** | Factor a common attribute out of an OR policy | Share a gate output across multiple logic paths |
| **DNF** | Read off every attribute combination that grants access | Truth table of a combinational circuit |
| **CNF** | Read off every constraint that must hold for denial | Clause form used in hardware verification |

**The central lesson of Module 01.** Propositional logic and Boolean algebra are
not separate things — they are the same algebra at two levels:

- **Software:** policy expressions evaluated at runtime by a policy engine
- **Hardware:** gate networks evaluated at nanosecond timescales by physical circuits

The bugs are the same too. A missing check in a policy is a missing gate in a
circuit. An MFA bypass is a short circuit. The tools to find them — truth tables,
algebraic simplification, normal forms — are identical at both levels.

---


## Python: Visualization & Verification

**1 · Boolean Algebra Law Verifier** — use SymPy to prove every law in the
reference table holds, and show the simplification of Example 2 step by step.

**2 · Logic Gate Circuit Visualizer** — draw the gate-level implementation of
the ABAC policy fragment $A \wedge \neg S$, connecting the hardware and software
views of the same Boolean expression.

**3 · MFA Bypass — Full Capstone Analysis** — compare the intended four-variable
policy against the simplified (broken) version, enumerate the blast radius, and
visualise the difference across all 16 input combinations.


In [ ]:
import subprocess, sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

for pkg in ["numpy", "matplotlib", "sympy", "scipy", "networkx"]:
    install(pkg)

print("All packages installed.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.colors import ListedColormap
import networkx as nx
import itertools
from sympy import symbols
from sympy.logic.boolalg import (
    And, Or, Not, Xor,
    to_dnf, to_cnf, simplify_logic, bool_map
)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'lines.linewidth': 2, 'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

def tf(b): return 'T' if b else 'F'

print('Setup complete.')


### 1 · Boolean Algebra Law Verifier and the MFA Bypass

SymPy's `simplify_logic` applies Boolean algebra laws automatically, returning
the simplest equivalent form it can find. We use it in two ways:

- **Verify every law in the table** — confirm each identity holds algebraically
- **Trace the MFA bypass** — show the step-by-step simplification that removes
  the MFA check from the four-variable policy


In [ ]:
# ── 1 · Law Verifier and MFA Bypass ──────────────────────────────────────────
A_s, T_s, S_s, M_s, P_s, Q_s, R_s = symbols('A T S M P Q R')

print("=" * 62)
print("BOOLEAN ALGEBRA LAW VERIFICATION (via SymPy simplify_logic)")
print("=" * 62)

laws = [
    ("Identity  (∧)",     And(P_s, True),             P_s),
    ("Identity  (∨)",     Or(P_s, False),              P_s),
    ("Null      (∧)",     And(P_s, False),             False),
    ("Null      (∨)",     Or(P_s, True),               True),
    ("Idempotent (∧)",    And(P_s, P_s),               P_s),
    ("Idempotent (∨)",    Or(P_s, P_s),                P_s),
    ("Double Neg",        Not(Not(P_s)),               P_s),
    ("Complement (∧)",    And(P_s, Not(P_s)),          False),
    ("Complement (∨)",    Or(P_s, Not(P_s)),           True),
    ("De Morgan 1",       Not(And(P_s, Q_s)),          Or(Not(P_s), Not(Q_s))),
    ("De Morgan 2",       Not(Or(P_s, Q_s)),           And(Not(P_s), Not(Q_s))),
    ("Distributive (∧)",  And(P_s, Or(Q_s, R_s)),
                          Or(And(P_s, Q_s), And(P_s, R_s))),
    ("Absorption (∧)",    And(P_s, Or(P_s, Q_s)),     P_s),
]

all_pass = True
for name, lhs, rhs in laws:
    lhs_s = simplify_logic(lhs)
    rhs_s = simplify_logic(rhs)
    ok    = simplify_logic(lhs_s) == simplify_logic(rhs_s)
    all_pass = all_pass and ok
    status = "✓" if ok else "✗"
    print(f"  {status}  {name:<22}  {str(lhs_s):<20}  ≡  {str(rhs_s)}")

print()
print(f"All laws verified: {'YES ✓' if all_pass else 'NO — check above'}")

# ── MFA Bypass step by step ────────────────────────────────────────────────────
print()
print("=" * 62)
print("MFA BYPASS — STEP-BY-STEP SIMPLIFICATION")
print("=" * 62)

correct  = And(A_s, T_s, Not(S_s), M_s)
broken   = Or(And(A_s, T_s, Not(S_s), M_s),
              And(A_s, T_s, Not(S_s), Not(M_s)))

steps = [
    ("Intended policy (with MFA)", correct),
    ("Developer's 'optimised' version", broken),
    ("After simplify_logic", simplify_logic(broken)),
    ("DNF of simplified", to_dnf(broken, simplify=True)),
]

for label, expr in steps:
    print(f"  {label}:")
    print(f"    {expr}")
    print()

equivalent_to_correct = simplify_logic(broken) == simplify_logic(correct)
print(f"  Broken policy ≡ correct policy?  {'YES — MFA still enforced' if equivalent_to_correct else 'NO — MFA has been REMOVED ⚠'}")


**What do you see?**

Every law in the table verifies — ✓ on every row. SymPy confirms the algebra
we have been using is sound.

The MFA bypass output shows the critical step: `simplify_logic` reduces the
developer's "optimised" expression to `A ∧ T ∧ ¬S` — exactly the three-variable
policy from Unit 02, with the MFA clause gone. The final line confirms:
**broken ≢ correct.** The two policies admit different users.

Notice how the step-by-step output mirrors the algebraic derivation in Section 2:
the `Or(And(..., M), And(..., Not(M)))` structure is precisely
$A \wedge T \wedge \neg S \wedge (M \vee \neg M)$, and SymPy applies the complement
law automatically.


### 2 · Logic Gate Circuit Visualizer

We now draw the gate-level implementation of the ABAC policy fragment
$A \wedge \neg S$ — *"admin and not suspended"* — as a logic circuit.

This is exactly how an access control policy would be evaluated in hardware:
the user's attributes arrive as input signals, pass through NOT and AND gates,
and the output signal drives the access decision. The circuit diagram makes the
physical realisation of Boolean algebra visible.


In [ ]:
# ── 2 · Logic Gate Circuit Visualizer ────────────────────────────────────────
#
# Circuit for: A ∧ ¬S  (admin AND not-suspended)
#
# Inputs: A (admin), S (suspended)
# Gates:  NOT on S, then AND(A, NOT_S)
# Output: allow

fig, ax = plt.subplots(figsize=(11, 5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_facecolor('white')
fig.patch.set_facecolor('white')

def wire(ax, x0, y0, x1, y1, color=None, lw=2):
    c = color or TS_GRAY
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='-', color=c, lw=lw))

def gate_box(ax, x, y, label, sublabel='', width=1.6, height=0.9,
             fc=TS_LIGHT, ec=TS_BLUE, fontsize=11):
    rect = mpatches.FancyBboxPatch(
        (x - width/2, y - height/2), width, height,
        boxstyle='round,pad=0.12', facecolor=fc, edgecolor=ec, linewidth=2
    )
    ax.add_patch(rect)
    ax.text(x, y + (0.1 if sublabel else 0), label,
            ha='center', va='center', fontsize=fontsize,
            fontweight='bold', color=TS_BLUE)
    if sublabel:
        ax.text(x, y - 0.22, sublabel, ha='center', va='center',
                fontsize=8, color=TS_GRAY)

def input_node(ax, x, y, label, value_label):
    circle = plt.Circle((x, y), 0.22, color=TS_AMBER, zorder=3)
    ax.add_patch(circle)
    ax.text(x, y, '', ha='center', va='center', fontsize=9, color='white')
    ax.text(x - 0.55, y, label, ha='right', va='center',
            fontsize=12, fontweight='bold', color=TS_BLUE)
    ax.text(x - 0.55, y - 0.35, value_label, ha='right', va='center',
            fontsize=9, color=TS_GRAY, style='italic')

def output_node(ax, x, y, label):
    rect = mpatches.FancyBboxPatch(
        (x - 0.55, y - 0.28), 1.1, 0.56,
        boxstyle='round,pad=0.08', facecolor=TS_GREEN, edgecolor='white', linewidth=2
    )
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')

# ── Inputs ────────────────────────────────────────────────────────────────────
input_node(ax, 1.2, 4.5, 'A', '(admin role)')
input_node(ax, 1.2, 2.0, 'S', '(suspended)')

# ── NOT gate on S ─────────────────────────────────────────────────────────────
gate_box(ax, 3.5, 2.0, 'NOT', '¬S', width=1.4, height=0.75,
         fc='#fff3e0', ec=TS_AMBER)

# ── AND gate ──────────────────────────────────────────────────────────────────
gate_box(ax, 6.5, 3.25, 'AND', 'A ∧ ¬S', width=1.6, height=0.9,
         fc='#e8f4fd', ec=TS_BLUE)

# ── Output ────────────────────────────────────────────────────────────────────
output_node(ax, 9.0, 3.25, 'allow?')

# ── Wires ─────────────────────────────────────────────────────────────────────
# A → AND (upper input)
wire(ax, 1.42, 4.5, 5.7, 3.55, color=TS_BLUE)
# S → NOT
wire(ax, 1.42, 2.0, 2.8, 2.0, color=TS_AMBER)
# NOT → AND (lower input)
wire(ax, 4.2, 2.0, 5.7, 2.95, color=TS_AMBER)
# AND → output
wire(ax, 7.3, 3.25, 8.45, 3.25, color=TS_GREEN, lw=2.5)

# ── Labels ────────────────────────────────────────────────────────────────────
ax.text(3.5, 1.3, 'NOT gate
Flips S → ¬S
(¬ connective)', ha='center',
        fontsize=8, color=TS_AMBER, style='italic')
ax.text(6.5, 2.4, 'AND gate
True iff both inputs T
(∧ connective)', ha='center',
        fontsize=8, color=TS_BLUE, style='italic')

# ── Title and truth table inset ───────────────────────────────────────────────
ax.set_title('Gate Circuit for $A \\wedge \\neg S$ — "Admin AND not suspended"
'
             'High voltage = T  ·  Low voltage = F  ·  Signal flows left → right',
             fontsize=11, pad=10)

# Small truth table
rows_tt = [(T, S, not S, T and not S)
           for T in [True, False] for S in [True, False]]
tt_x, tt_y = 8.2, 5.5
ax.text(tt_x, tt_y, 'A   S   ¬S  out', ha='center', fontsize=8,
        color=TS_GRAY, fontfamily='monospace')
for i, (a_v, s_v, ns_v, out_v) in enumerate(rows_tt):
    color = TS_GREEN if out_v else TS_RED
    ax.text(tt_x, tt_y - 0.32*(i+1),
            f'{tf(a_v)}   {tf(s_v)}    {tf(ns_v)}    {tf(out_v)}',
            ha='center', fontsize=8, color=color, fontfamily='monospace')

plt.tight_layout()
plt.show()


**What do you see?**

The circuit diagram shows three components connected by wires:

1. **Input nodes** (amber circles): $A$ and $S$ carry the user's attribute values
   as signals — high voltage for True, low for False.
2. **NOT gate** (orange box): inverts the $S$ signal to produce $\neg S$.
   This is the suspension check — a suspended user's signal is flipped to False
   before it reaches the AND gate.
3. **AND gate** (blue box): outputs True only when *both* inputs are True —
   the access decision fires only for an unsuspended admin.

The **inset truth table** on the right confirms the circuit's four possible states.
Only the top-right row (A=T, S=F) produces a high output signal.

**The hardware/software connection:** The access control bug we studied in Unit 02
— where $S$ defaults to False — corresponds to **cutting the wire between the
input node $S$ and the NOT gate**. The NOT gate still exists, but it receives a
constant low signal (False) and outputs a constant high (True). The AND gate sees
$\neg S = \top$ permanently. The missing check is a literal short circuit.


### 3 · MFA Bypass — Full Capstone Analysis

We now run the complete four-variable analysis. The **intended policy** requires
all four attributes:

$$\text{allow\_correct} \;\equiv\; A \wedge T \wedge \neg S \wedge M$$

The **broken policy** — the developer's "exemption" that removes MFA:

$$\text{allow\_broken} \;\equiv\; (A \wedge T \wedge \neg S \wedge M) \;\vee\; (A \wedge T \wedge \neg S \wedge \neg M)$$

With four variables there are $2^4 = 16$ input combinations. We generate the full
truth table, compare both policies row by row, highlight the exposed rows,
and visualise the DNF of each policy as a heatmap over the input space.


In [ ]:
# ── 3 · MFA Bypass — Full Four-Variable Capstone ─────────────────────────────

# Policies as Python lambdas
correct = lambda A,T,S,M: A and T and not S and M
broken  = lambda A,T,S,M: (A and T and not S and M) or (A and T and not S and not M)

combos   = list(itertools.product([True, False], repeat=4))
variables = ['A','T','S','M']

results = []
for vals in combos:
    A,T,S,M = vals
    c = correct(A=A, T=T, S=S, M=M)
    b = broken(A=A,  T=T, S=S, M=M)
    exposed = b and not c
    results.append((*vals, c, b, exposed))

# ── Summary stats ─────────────────────────────────────────────────────────────
correct_grants = sum(r[4] for r in results)
broken_grants  = sum(r[5] for r in results)
exposed_count  = sum(r[6] for r in results)

print(f"Total input combinations (2⁴): {len(results)}")
print(f"Correct policy grants access:  {correct_grants}/16  "
      f"(only: admin, in-window, not-suspended, MFA-verified)")
print(f"Broken policy grants access:   {broken_grants}/16  "
      f"(MFA removed — admin, in-window, not-suspended)")
print(f"Exposed by the bug:            {exposed_count} row(s)")
print()

print("Exposed rows (correct=DENY, broken=ALLOW):")
for A,T,S,M,c,b,exp in results:
    if exp:
        print(f"  A={tf(A)}, T={tf(T)}, S={tf(S)}, M={tf(M)}"
              f"  →  admin, in-window, NOT-suspended, NO MFA")
print()

# ── SymPy DNF confirmation ─────────────────────────────────────────────────────
broken_sym  = Or(And(A_s, T_s, Not(S_s), M_s),
                 And(A_s, T_s, Not(S_s), Not(M_s)))
correct_sym = And(A_s, T_s, Not(S_s), M_s)

print("SymPy DNF of broken policy: ", to_dnf(broken_sym,  simplify=True))
print("SymPy DNF of correct policy:", to_dnf(correct_sym, simplify=True))
print()
print("Broken simplifies to:", simplify_logic(broken_sym))
print("MFA check removed?   ", simplify_logic(broken_sym) != correct_sym)

# ── Heatmap over all 16 combinations ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 5.5))

# For each heatmap: rows = (A,T,S), cols = M
ats_combos = list(itertools.product([True, False], repeat=3))
m_vals     = [True, False]

for ax, col_idx, title, policy_fn in [
    (axes[0], 4, 'Correct policy
$A \\wedge T \\wedge \\neg S \\wedge M$', correct),
    (axes[1], 5, 'Broken policy
$(A \\wedge T \\wedge \\neg S \\wedge M) \\vee (A \\wedge T \\wedge \\neg S \\wedge \\neg M)$', broken),
    (axes[2], 6, 'Exposed by bug
(broken=ALLOW, correct=DENY)', None),
]:
    matrix = np.zeros((len(ats_combos), len(m_vals)))
    for r, (A,T,S) in enumerate(ats_combos):
        for c, M in enumerate(m_vals):
            if policy_fn:
                matrix[r, c] = float(policy_fn(A=A, T=T, S=S, M=M))
            else:
                matrix[r, c] = float(broken(A=A,T=T,S=S,M=M)
                                      and not correct(A=A,T=T,S=S,M=M))

    if col_idx == 6:
        cmap = ListedColormap(['#f8f9fa', TS_RED])
    else:
        cmap = ListedColormap([TS_RED, TS_GREEN])

    ax.imshow(matrix, cmap=cmap, vmin=0, vmax=1, aspect='auto')

    for r in range(len(ats_combos)):
        for c in range(len(m_vals)):
            val = bool(matrix[r, c])
            if col_idx == 6:
                txt   = '⚠' if val else '—'
                color = 'white' if val else TS_GRAY
            else:
                txt   = 'T' if val else 'F'
                color = 'white'
            ax.text(c, r, txt, ha='center', va='center',
                    fontsize=12, fontweight='bold', color=color)

    row_labels = [f'A={tf(A)},T={tf(T)},S={tf(S)}' for A,T,S in ats_combos]
    ax.set_yticks(range(len(ats_combos)))
    ax.set_yticklabels(row_labels, fontsize=8)
    ax.set_xticks(range(len(m_vals)))
    ax.set_xticklabels([f'M={tf(m)}' for m in m_vals], fontsize=9)
    ax.set_title(title, fontsize=9, fontweight='bold', pad=8)
    ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)

plt.suptitle('Four-Variable ABAC Policy: Correct vs Broken vs Exposed',
             fontsize=12, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()


**What do you see?**

The three heatmaps tell the complete story:

- **Correct policy** (left): exactly one green cell — $A=T, T=T, S=F, M=T$.
  The admin, in-window, not-suspended, MFA-verified user. One cell out of sixteen.

- **Broken policy** (centre): two green cells — the same row plus $M=F$.
  The MFA column no longer matters; both values grant access.

- **Exposed** (right): the single ⚠ cell — $A=T, T=T, S=F, M=F$.
  The admin who bypassed MFA. One extra row. One missing gate. One algebraic
  simplification that the developer did not notice.

**The broader lesson.** Four variables means 16 rows — small enough to check by
hand. A real enterprise ABAC policy might have 15–20 attribute clauses, producing
$2^{15}$ to $2^{20}$ rows. At that scale, manual truth table verification is
impossible. Boolean algebra simplification and SAT solvers — both built on the
foundations in this module — are the only practical tools. This is why formal
methods matter in security engineering, and why the mathematics is not optional.


In [ ]:
# ── Extension Challenge ───────────────────────────────────────────────────────
#
# 1. NAND UNIVERSALITY
#    Implement NOT, AND, and OR using only NAND operations in Python:
#      nand = lambda A, B: not (A and B)
#      my_not(P)    = ?
#      my_and(P, Q) = ?
#      my_or(P, Q)  = ?   (Hint: use De Morgan's law)
#    Verify each with a truth table.
#
# 2. CNF CONVERSION
#    Use SymPy's to_cnf to convert the correct four-variable ABAC policy
#    to CNF. How many clauses are there? What does each clause represent
#    in access-denial terms?
#
# 3. FIVE-VARIABLE POLICY
#    Add a fifth attribute: L = "login is from a known IP" (location check).
#    Define correct5 = A ∧ T ∧ ¬S ∧ M ∧ L.
#    Suppose a developer writes:
#      broken5 = (correct5) ∨ (A ∧ T ∧ ¬S ∧ M ∧ ¬L)
#    Use simplify_logic to determine what this simplifies to.
#    What is the blast radius? Which variable's check was removed?

# Your work here:


---

## Summary

| Concept | Definition | Security application |
|---|---|---|
| **Boolean algebra laws** | Rewrite rules preserving equivalence | Simplify access policies without truth tables |
| **Algebraic simplification** | Apply laws step by step to reduce a formula | Detect missing checks, redundant clauses |
| **DNF** | OR of AND-terms — one term per satisfying row | Enumerate every path to access |
| **CNF** | AND of OR-clauses — one clause per denying row | Enumerate every denial constraint |
| **Logic gates** | Hardware realisation of $\neg$, $\wedge$, $\vee$, $\oplus$ | MAC units, combinational circuits, TPUs |
| **Complement law** | $P \vee \neg P \equiv \top$ — the silent eraser | Vacuous exemption removes a security check |
| **MFA bypass** | OR of two complementary terms collapses to $\top$ | One algebraic step removes a control |

---

## Module 01 Complete

You have now worked through the full arc of Module 01:

| Unit | Content | Payoff |
|---|---|---|
| **01** | Five connectives, precedence | Read and write access control formulas |
| **02** | Truth tables, equivalence, De Morgan | Verify policies exhaustively; detect missing checks |
| **03** | Boolean algebra, normal forms, gates | Simplify policies algebraically; catch bugs pre-implementation |

The three units form one argument: **access control policy is Boolean algebra,
Boolean algebra has a hardware realisation, and both layers are vulnerable to the
same class of formal error.** The complement law does not care whether it erases
an MFA check in software or a signal in a circuit.

---

## Up Next

**Module 02 — Predicate Logic and Quantifiers**

We move beyond propositions about fixed values to statements that range over
entire sets of inputs. Predicate logic gives us the formal language to write:

$$\forall x \in \mathcal{I},\; \text{validate}(x) = \mathtt{true} \Rightarrow \text{safe}(x) = \mathtt{true}$$

and to prove or disprove it — the foundation of reasoning about *all possible
inputs* to an AI system, not just the cases you thought to test.

$\rightarrow$ `modules/module-02/unit-01-predicates-quantifiers.ipynb`
